# 📣 BrandSphere AI — 04: Campaign KPI Predictor
**Module 3: Smart Social & Brand Campaign Studio**
Trains Random Forest models to predict CTR, ROI, and Engagement Score

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle, warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

## Step 1: Load & Prepare Data

In [ ]:
platforms = ['Instagram','Facebook','Twitter/X','LinkedIn','TikTok']
objectives = ['Brand Awareness','Engagement','Conversion','Lead Generation']
regions = ['North America','Europe','Asia','Middle East','Africa']
industries = ['Tech','Fashion','Food','Finance','Health']

n = 2000
df = pd.DataFrame({
    'platform': np.random.choice(platforms, n),
    'objective': np.random.choice(objectives, n),
    'region': np.random.choice(regions, n),
    'industry': np.random.choice(industries, n),
    'budget_usd': np.random.randint(500, 50000, n),
    'duration_days': np.random.randint(7, 90, n),
})

# Realistic target variables with feature influence
platform_ctr = {'Instagram':4.5,'Facebook':3.2,'Twitter/X':2.8,'LinkedIn':2.1,'TikTok':5.1}
obj_roi = {'Brand Awareness':20,'Engagement':45,'Conversion':120,'Lead Generation':85}

df['ctr_percent'] = (df['platform'].map(platform_ctr) +
                     np.log1p(df['budget_usd'])*0.1 +
                     np.random.normal(0, 0.5, n)).clip(0.5, 10).round(2)
df['roi_percent'] = (df['objective'].map(obj_roi) +
                     df['duration_days']*0.5 +
                     np.random.normal(0, 15, n)).clip(-10, 400).round(1)
df['engagement_score'] = (df['ctr_percent']*8 +
                           np.log1p(df['budget_usd'])*2 +
                           np.random.normal(0, 5, n)).clip(10, 100).round(1)

print(f'Dataset: {df.shape}')
df[['ctr_percent','roi_percent','engagement_score']].describe()

## Step 2: Feature Engineering & Encoding

In [ ]:
le_dict = {}
cat_cols = ['platform','objective','region','industry']
df_encoded = df.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[f'{col}_enc'] = le.fit_transform(df[col])
    le_dict[col] = le

scaler = MinMaxScaler()
df_encoded[['budget_scaled','duration_scaled']] = scaler.fit_transform(df[['budget_usd','duration_days']])

features = [f'{c}_enc' for c in cat_cols] + ['budget_scaled','duration_scaled']
X = df_encoded[features]

print(f'Feature matrix: {X.shape}')
print(f'Features: {features}')

## Step 3: Train Random Forest Models (CTR, ROI, Engagement)

In [ ]:
targets = {'ctr_percent': df['ctr_percent'], 'roi_percent': df['roi_percent'], 'engagement_score': df['engagement_score']}
models = {}
results = {}

for target_name, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_split=5,
                                random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    cv_r2 = cross_val_score(rf, X, y, cv=5, scoring='r2').mean()

    models[target_name] = rf
    results[target_name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'CV_R2': cv_r2}
    print(f'{target_name.upper():20s} | RMSE: {rmse:.3f} | MAE: {mae:.3f} | R²: {r2:.4f} | CV R²: {cv_r2:.4f}')

In [ ]:
# ── Feature Importance Plot ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
feature_labels = ['Platform','Objective','Region','Industry','Budget','Duration']
colors = ['#2E86AB','#F18F01','#44BBA4']

for ax, (name, model), color in zip(axes, models.items(), colors):
    importances = model.feature_importances_
    sorted_idx = np.argsort(importances)
    ax.barh([feature_labels[i] for i in sorted_idx], importances[sorted_idx], color=color)
    ax.set_title(f'{name.replace("_"," ").title()}\nFeature Importance', fontweight='bold')
    ax.set_xlabel('Importance')

plt.tight_layout()
plt.savefig('campaign_feature_importance.png', dpi=150)
plt.show()

In [ ]:
# ── Predicted vs Actual Scatter ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, model), color in zip(axes, models.items(), ['#2E86AB','#F18F01','#44BBA4']):
    y = targets[name]
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    y_pred = model.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.3, color=color, s=10)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect')
    ax.set_title(f'{name.replace("_"," ").title()}\nR²={results[name]["R2"]:.3f}', fontweight='bold')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.legend()
plt.tight_layout()
plt.savefig('campaign_predicted_vs_actual.png', dpi=150)
plt.show()

## Step 4: Save Models

In [ ]:
import os
os.makedirs('../config', exist_ok=True)
with open('../config/campaign_models.pkl', 'wb') as f:
    pickle.dump({'models': models, 'encoders': le_dict, 'scaler': scaler, 'features': features}, f)

print('Campaign models saved: ../config/campaign_models.pkl')
print('\n=== RESULTS SUMMARY ===')
for name, r in results.items():
    print(f'{name:20s}: R²={r["R2"]:.4f} | RMSE={r["RMSE"]:.3f} | MAE={r["MAE"]:.3f}')
print('\nNext: Run 05_retraining.ipynb')